# Laser / Live Mode Diagnostic
Step through cells one by one. Each cell prints what it does and what changed.

In [ ]:
from pycromanager import Core
core = Core()
print('Connected')
print('Active camera  :', core.get_camera_device())
print('Active shutter :', core.get_shutter_device())
print('Auto-shutter   :', core.get_auto_shutter())

## 1. Read current state

In [ ]:
PLOGIC = 'PLogic:E:36'
CAM1   = 'Kinetix22-1'
CAM2   = 'Kinetix22-2'

# Probe common trigger property names directly — no iteration needed
for prop in ['TriggerMode', 'Trigger Mode', 'TriggerSource', 'Trigger',
             'Timing-TriggerMode', 'ExposureMode', 'ReadoutMode']:
    try:
        val = core.get_property(CAM1, prop)
        print(f'FOUND  {prop!r:30s} = {val!r}')
    except Exception:
        print(f'       {prop!r:30s}   (not found)')

print()
print('PLogic OutputChannel:', core.get_property(PLOGIC, 'OutputChannel'))
print('Auto-shutter        :', core.get_auto_shutter())

In [ ]:
PLOGIC   = 'PLogic:E:36'
CAM1     = 'Kinetix22-1'
CAM2     = 'Kinetix22-2'

print('--- PLogic ---')
print('OutputChannel  :', core.get_property(PLOGIC, 'OutputChannel'))

print()
print('--- Kinetix22-1 ---')
print('TriggerMode    :', core.get_property(CAM1, 'TriggerMode'))

print()
print('--- Kinetix22-2 ---')
print('TriggerMode    :', core.get_property(CAM2, 'TriggerMode'))

## 2. Force Internal trigger + set OutputChannel
Edit `OUTPUT_CHANNEL` to match what works when you set it manually in Device Property Browser.

In [ ]:
# ▶ RUN THIS BEFORE GOING LIVE
core.set_property(CAM1, 'TriggerMode', 'Internal Trigger')
core.set_property(CAM2, 'TriggerMode', 'Internal Trigger')
core.set_property(PLOGIC, 'OutputChannel', 'output 3 only')
core.set_auto_shutter(False)
core.snap_image(); _ = core.get_tagged_image()   # prime PLogic latch
core.set_shutter_open(True)
print('Ready for Live — laser will be ON.')

In [ ]:
# Run this immediately AFTER stopping Live — shows what MM reset
print('--- State after Live stopped ---')
print('TriggerMode CAM1 :', core.get_property(CAM1, 'TriggerMode'))
print('TriggerMode CAM2 :', core.get_property(CAM2, 'TriggerMode'))
print('OutputChannel    :', core.get_property(PLOGIC, 'OutputChannel'))
print('Auto-shutter     :', core.get_auto_shutter())
print('Shutter open     :', core.get_shutter_open())

## 3. Check what OutputChannel values are available

In [ ]:
vals = core.get_allowed_property_values(PLOGIC, 'OutputChannel')
print('Allowed OutputChannel values:')
for v in vals:
    print(' ', v)

## 4. Check what TriggerMode values are available on Kinetix22-1

In [ ]:
vals = core.get_allowed_property_values(CAM1, 'TriggerMode')
print('Allowed TriggerMode values (Kinetix22-1):')
for v in vals:
    print(' ', v)

## 5. Disable auto-shutter and open shutter manually
If the above didn't fix it, try disabling auto-shutter.

In [ ]:
core.set_auto_shutter(False)
core.set_shutter_open(True)
print('Auto-shutter OFF, shutter OPEN')
print('Now go Live in MM — laser should be ON.')
print()
print('To restore:')
print('  core.set_shutter_open(False)')
print('  core.set_auto_shutter(True)')

## 6. Restore auto-shutter

In [ ]:
# ▶ RUN THIS AFTER LIVE — RESTORES STATE FOR LSM PLUGIN
core.set_shutter_open(False)
core.set_auto_shutter(True)
core.set_property(PLOGIC, 'OutputChannel', 'none of outputs 1-7')
print('Restored for LSM — auto-shutter ON, laser OFF.')